In [3]:
"""
Latent Factor Analysis — Malignancy Probabilities
====================================================
Collapses 30 features into 5 clinically interpretable latent categories,
then computes P(Malignant) when each factor shows a positive signal.
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report
from sklearn.calibration import CalibratedClassifierCV
import warnings
warnings.filterwarnings("ignore")

# Set for Output Directory (adjust this path as needed)

from pathlib import Path

OUTPUT_DIR = Path("/Users/rudolphsurovcik/Library/CloudStorage/OneDrive-Personal/Python Folder")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Then every save call becomes:
#plt.savefig(OUTPUT_DIR / "02_permutation_importance.png", dpi=150, bbox_inches="tight")
#plt.savefig(OUTPUT_DIR / "03_shap_beeswarm.png", dpi=150, bbox_inches="tight")


plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f8f9fa",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.family": "sans-serif",
    "font.size": 11,
})

# ═══════════════════════════════════════════════════════════════════════════════
# 1. LOAD DATA & DEFINE LATENT CATEGORIES
# ═══════════════════════════════════════════════════════════════════════════════
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
# Flip target: 1 = malignant, 0 = benign (original is reversed)
df["malignant"] = 1 - data.target

print("=" * 72)
print("LATENT FACTOR ANALYSIS — MALIGNANCY PROBABILITIES")
print("=" * 72)

# Define the five latent groupings
latent_groups = {
    "Tumor Size": [
        "mean radius", "mean perimeter", "mean area",
        "worst radius", "worst perimeter", "worst area"
    ],
    "Boundary Irregularity": [
        "mean concave points", "mean concavity",
        "worst concave points", "worst concavity"
    ],
    "Texture": [
        "mean texture", "worst texture"
    ],
    "Measurement Variability": [
        "area error", "radius error", "perimeter error"
    ],
    "Surface Morphology": [
        "mean smoothness", "mean compactness", "mean symmetry",
        "mean fractal dimension",
        "worst smoothness", "worst compactness", "worst symmetry",
        "worst fractal dimension",
        "smoothness error", "compactness error", "symmetry error",
        "fractal dimension error", "concavity error", "concave points error"
    ],
}

print("\nLatent Factor Definitions:")
for name, features in latent_groups.items():
    print(f"  {name:.<30s} {len(features)} features")

# ═══════════════════════════════════════════════════════════════════════════════
# 2. BUILD LATENT FACTORS VIA PCA (first principal component per group)
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 72)
print("BUILDING LATENT FACTORS (PCA — 1st component per group)")
print("─" * 72)

scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df.drop("malignant", axis=1)),
    columns=data.feature_names
)

latent_df = pd.DataFrame()
variance_explained = {}

for name, features in latent_groups.items():
    pca = PCA(n_components=1)
    component = pca.fit_transform(df_scaled[features])
    # Orient so that higher values = more concerning (correlate with malignancy)
    corr_with_malignant = np.corrcoef(component.ravel(), df["malignant"])[0, 1]
    if corr_with_malignant < 0:
        component = -component
    latent_df[name] = component.ravel()
    variance_explained[name] = pca.explained_variance_ratio_[0]
    print(f"  {name:.<30s} {pca.explained_variance_ratio_[0]*100:.1f}% variance captured")

latent_df["malignant"] = df["malignant"].values

# ═══════════════════════════════════════════════════════════════════════════════
# 3. TRAIN CALIBRATED MODEL ON LATENT FACTORS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 72)
print("TRAINING CALIBRATED RANDOM FOREST ON 5 LATENT FACTORS")
print("─" * 72)

X_latent = latent_df.drop("malignant", axis=1)
y = latent_df["malignant"]

X_train, X_test, y_train, y_test = train_test_split(
    X_latent, y, test_size=0.25, random_state=42, stratify=y
)

# Use calibrated classifier for reliable probability estimates
rf = RandomForestClassifier(
    n_estimators=300, min_samples_split=5, min_samples_leaf=2,
    class_weight="balanced", random_state=42, n_jobs=-1
)
cal_rf = CalibratedClassifierCV(rf, cv=5, method="isotonic")
cal_rf.fit(X_train, y_train)

train_acc = cal_rf.score(X_train, y_train)
test_acc = cal_rf.score(X_test, y_test)
cv_scores = cross_val_score(cal_rf, X_latent, y, cv=5, scoring="accuracy")

print(f"\n  Train accuracy : {train_acc:.4f}")
print(f"  Test accuracy  : {test_acc:.4f}")
print(f"  5-Fold CV      : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

print(f"\n  Classification Report (Test Set):")
y_pred = cal_rf.predict(X_test)
report = classification_report(y_test, y_pred, target_names=["Benign", "Malignant"])
for line in report.split("\n"):
    print(f"    {line}")

# ═══════════════════════════════════════════════════════════════════════════════
# 4. PROBABILITY ANALYSIS — POSITIVE SIGNAL ON EACH LATENT
# ═══════════════════════════════════════════════════════════════════════════════
print("─" * 72)
print("P(MALIGNANT) WHEN EACH LATENT FACTOR SHOWS A POSITIVE SIGNAL")
print("─" * 72)
print("\n  'Positive signal' = factor value in the top quartile (75th percentile+)")
print("  'No signal'       = factor value in the bottom quartile (25th percentile-)\n")

factor_names = list(latent_groups.keys())
results = []

# Baseline: all factors at median
median_profile = X_latent.median().values.reshape(1, -1)
baseline_prob = cal_rf.predict_proba(median_profile)[0, 1]
print(f"  {'Baseline (all factors at median)':<45s}  P(Malignant) = {baseline_prob:.1%}")
print()

for i, factor in enumerate(factor_names):
    # Positive signal: this factor at 75th percentile, others at median
    high_profile = X_latent.median().values.copy()
    high_profile[i] = X_latent[factor].quantile(0.75)
    p_high = cal_rf.predict_proba(high_profile.reshape(1, -1))[0, 1]

    # Strong positive signal: this factor at 90th percentile
    very_high_profile = X_latent.median().values.copy()
    very_high_profile[i] = X_latent[factor].quantile(0.90)
    p_very_high = cal_rf.predict_proba(very_high_profile.reshape(1, -1))[0, 1]

    # No signal: this factor at 25th percentile
    low_profile = X_latent.median().values.copy()
    low_profile[i] = X_latent[factor].quantile(0.25)
    p_low = cal_rf.predict_proba(low_profile.reshape(1, -1))[0, 1]

    results.append({
        "Factor": factor,
        "P(Mal) Low Signal": p_low,
        "P(Mal) Elevated": p_high,
        "P(Mal) Strongly Elevated": p_very_high,
        "Lift (Elevated vs Low)": p_high / max(p_low, 0.001),
    })

    print(f"  {factor:<30s}  Low: {p_low:.1%}   Elevated: {p_high:.1%}   Strongly Elevated: {p_very_high:.1%}")

# ═══════════════════════════════════════════════════════════════════════════════
# 5. COMBINATION ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 72)
print("P(MALIGNANT) FOR COMBINED FACTOR PROFILES")
print("─" * 72)

scenarios = {
    "All factors low (25th pctl)": dict.fromkeys(factor_names, 0.25),
    "All factors at median": dict.fromkeys(factor_names, 0.50),
    "Size elevated alone": {"Tumor Size": 0.75, "Boundary Irregularity": 0.50,
                            "Texture": 0.50, "Measurement Variability": 0.50,
                            "Surface Morphology": 0.50},
    "Size + Irregularity elevated": {"Tumor Size": 0.75, "Boundary Irregularity": 0.75,
                                     "Texture": 0.50, "Measurement Variability": 0.50,
                                     "Surface Morphology": 0.50},
    "Size + Irregularity + Texture": {"Tumor Size": 0.75, "Boundary Irregularity": 0.75,
                                      "Texture": 0.75, "Measurement Variability": 0.50,
                                      "Surface Morphology": 0.50},
    "Size + Irregularity + Variability": {"Tumor Size": 0.75, "Boundary Irregularity": 0.75,
                                          "Texture": 0.50, "Measurement Variability": 0.75,
                                          "Surface Morphology": 0.50},
    "All factors elevated (75th)": dict.fromkeys(factor_names, 0.75),
    "All factors strongly elevated (90th)": dict.fromkeys(factor_names, 0.90),
}

print()
combo_results = []
for label, quantiles in scenarios.items():
    profile = np.array([X_latent[f].quantile(quantiles[f]) for f in factor_names]).reshape(1, -1)
    prob = cal_rf.predict_proba(profile)[0, 1]
    combo_results.append({"Scenario": label, "P(Malignant)": prob})
    print(f"  {label:<45s}  P(Malignant) = {prob:.1%}")

# ═══════════════════════════════════════════════════════════════════════════════
# 6. VISUALIZATION
# ═══════════════════════════════════════════════════════════════════════════════

# ── Figure 1: Individual Factor Probabilities ────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))

factors = [r["Factor"] for r in results]
p_low = [r["P(Mal) Low Signal"] for r in results]
p_elev = [r["P(Mal) Elevated"] for r in results]
p_strong = [r["P(Mal) Strongly Elevated"] for r in results]

x = np.arange(len(factors))
width = 0.25

bars1 = ax.bar(x - width, p_low, width, label="Low (25th pctl)", color="#93c5fd", edgecolor="white")
bars2 = ax.bar(x, p_elev, width, label="Elevated (75th pctl)", color="#2563eb", edgecolor="white")
bars3 = ax.bar(x + width, p_strong, width, label="Strongly Elevated (90th pctl)", color="#1e3a5f", edgecolor="white")

ax.axhline(y=baseline_prob, color="#e11d48", linestyle="--", linewidth=1.5, label=f"Baseline (median) = {baseline_prob:.1%}")
ax.set_ylabel("P(Malignant)", fontsize=12)
ax.set_title("Probability of Malignancy by Latent Factor Signal Strength", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(factors, fontsize=10, rotation=15, ha="right")
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.legend(loc="upper left", fontsize=10)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.015, f"{h:.0%}",
                ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "latent_factor_probabilities.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print("\nSaved → latent_factor_probabilities.png")

# ── Figure 2: Scenario Cascade ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))

scenario_labels = [r["Scenario"] for r in combo_results]
scenario_probs = [r["P(Malignant)"] for r in combo_results]

colors = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, len(scenario_labels)))
bars = ax.barh(range(len(scenario_labels)), scenario_probs, color=colors, edgecolor="white", height=0.7)

ax.set_yticks(range(len(scenario_labels)))
ax.set_yticklabels(scenario_labels, fontsize=10)
ax.set_xlabel("P(Malignant)", fontsize=12)
ax.set_title("Malignancy Probability by Combined Factor Profile", fontsize=14, fontweight="bold")
ax.set_xlim(0, 1.1)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.invert_yaxis()

for bar, prob in zip(bars, scenario_probs):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f"{prob:.1%}", va="center", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "latent_scenario_cascade.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print("Saved → latent_scenario_cascade.png")

# ── Figure 3: Latent Factor Distributions by Diagnosis ──────────────────────
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle("Latent Factor Distributions — Benign vs. Malignant", fontsize=14, fontweight="bold")

for ax, factor in zip(axes, factor_names):
    benign_vals = latent_df[latent_df["malignant"] == 0][factor]
    malign_vals = latent_df[latent_df["malignant"] == 1][factor]
    ax.hist(benign_vals, bins=25, alpha=0.6, color="#2563eb", label="Benign", density=True)
    ax.hist(malign_vals, bins=25, alpha=0.6, color="#e11d48", label="Malignant", density=True)
    ax.set_title(factor, fontsize=10, fontweight="bold")
    ax.set_xlabel("")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "latent_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print("Saved → latent_distributions.png")

# ═══════════════════════════════════════════════════════════════════════════════
# 7. SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 72)
print("SUMMARY")
print("=" * 72)
print(f"""
  Model: Calibrated Random Forest on 5 latent factors
  Accuracy: {test_acc:.1%} (vs. {0.958:.1%} on original 30 features)
  
  Key Finding: Collapsing 30 features → 5 latent factors preserves 
  nearly all predictive power while making the model interpretable.
  
  The two dominant drivers of malignancy probability:
    1. Tumor Size — the single most powerful predictor
    2. Boundary Irregularity — the strongest independent signal
  
  When both are elevated, P(Malignant) rises sharply.
  Adding Texture and Variability further increases confidence.
""")
print("Done.")

LATENT FACTOR ANALYSIS — MALIGNANCY PROBABILITIES

Latent Factor Definitions:
  Tumor Size.................... 6 features
  Boundary Irregularity......... 4 features
  Texture....................... 2 features
  Measurement Variability....... 3 features
  Surface Morphology............ 14 features

────────────────────────────────────────────────────────────────────────
BUILDING LATENT FACTORS (PCA — 1st component per group)
────────────────────────────────────────────────────────────────────────
  Tumor Size.................... 97.6% variance captured
  Boundary Irregularity......... 89.8% variance captured
  Texture....................... 95.6% variance captured
  Measurement Variability....... 96.9% variance captured
  Surface Morphology............ 49.0% variance captured

────────────────────────────────────────────────────────────────────────
TRAINING CALIBRATED RANDOM FOREST ON 5 LATENT FACTORS
────────────────────────────────────────────────────────────────────────

  Train acc